# A Self-Interpreter for SKI, Written in SKI

618 atoms over the basis `{S, K, I}`. No definitions, no host support, all
recursion through `Y`. It reduces SKI terms, and it reduces itself reducing
SKI terms.

This notebook builds it from nothing and runs it. Everything below executes;
nothing is quoted from elsewhere.

    pip install aviary-kernel==0.1.0

**Why this is not trivial.** A self-interpreter must take a program apart —
find the head, collect the arguments, decide which rule applies. But SKI
cannot inspect its own terms. Jay & Given-Wilson (*A combinatory account of
internal structure*, JSL 76(3), 2011, Thm 3.2) proved that factorisation —
recovering `X` from `S K X` — is **not representable** by any SK-combinator.
A calculus of pure functions can only *apply* its argument, never look at it.

There are two ways to pay for the missing capability. Add it to the calculus:
Jay & Palsberg (ICFP 2011) extend SK with a factorisation operator `F` and get
a typed self-interpreter in 1185 operators. Or move it into the *encoding*:
represent a term as a function that dispatches on its own shape. That is the
route here, and it costs nothing but size.

## 1. Setup

One piece of machinery needs explaining. The stock reducer checks term size on
every contraction, and reduction produces a **DAG** — `S x y z → x z (y z)`
duplicates `z` by *reference*. A compact DAG can denote an exponentially larger
tree, so the size check walks an exponential expansion and costs far more than
the reduction itself. The `run` below is the stock normal-order reducer with
that check removed; fuel is the guard instead.

In [1]:
import sys, time
sys.setrecursionlimit(10_000_000)
from aviary_kernel.environment import Environment
from aviary_kernel.terms import Atom, App, pretty, size, apply as _apply
from aviary_kernel.reduce import Status, _decompose, _try_contract, ReduceResult, _Frame
from aviary_kernel.abstraction import expand

env = Environment()
def A(*ts):
    r = ts[0]
    for t in ts[1:]: r = App(r, t)
    return r
a = v = Atom
def D(name, formals, body):
    env.define_rule(name, tuple(formals), body); return a(name)
def AL(name, body):
    env.define_alias(name, body); return a(name)
S, K, I, Y = a("S"), a("K"), a("I"), a("Y")

def dag_size(term):
    seen, stack, n = set(), [term], 0
    while stack:
        x = stack.pop()
        if id(x) in seen: continue
        seen.add(id(x)); n += 1
        if isinstance(x, App): stack.append(x.fn); stack.append(x.arg)
    return n

def run(term, *, whnf=True, fuel=10**8):
    steps, stack = 0, []
    def full(head, args):
        r = _apply(head, *args)
        for fr in reversed(stack):
            rem = fr.todo[len(fr.done)+1:]
            r = _apply(fr.head, *(fr.done + [r] + rem))
        return r
    head, args = _decompose(term)
    while True:
        while True:
            if steps >= fuel:
                return ReduceResult(full(head, args), steps, Status.FUEL, [])
            c = _try_contract(head, args, env)
            if c is None: break
            head, args, _ = c; steps += 1
        if whnf and not stack:
            return ReduceResult(_apply(head, *args), steps, Status.WHNF, [])
        if args:
            stack.append(_Frame(head=head, done=[], todo=list(args)))
            head, args = _decompose(args[0]); continue
        res = head
        while stack:
            fr = stack[-1]; fr.done.append(res)
            rem = fr.todo[len(fr.done):]
            if rem:
                head, args = _decompose(rem[0]); break
            built = _apply(fr.head, *fr.done); stack.pop(); res = built
        else:
            return ReduceResult(res, steps, Status.NORMAL, [])
print("ready")

ready


## 2. The problem, concretely

`S K K` and `I` are the same function. Every context in SKI treats them
identically — that is what extensionality means. Yet they are different
terms, 3 atoms and 1, and an interpreter must be able to tell them apart.

In [2]:
skk = A(S, K, K)
print("S K K x  ->", pretty(run(A(skk, v("x")), whnf=False).term))
print("I x      ->", pretty(run(A(I, v("x")), whnf=False).term))
print("same function, different terms:", pretty(skk), "vs", pretty(I))

S K K x  -> x
I x      -> x
same function, different terms: S K K vs I


## 3. Scott encoding: make the term dispatch on itself

If SKI cannot inspect a term, encode the term as something that inspects
*itself*. A Scott-encoded constructor takes one continuation per constructor
and applies the matching one:

    <S>    = λ s k i z. s
    <K>    = λ s k i z. k
    <I>    = λ s k i z. i
    <t u>  = λ s k i z. z <t> <u>

Case analysis becomes a single application. The intensional power we lacked
is manufactured by the representation.

In [3]:
D("encS", ["s","k","i","z"], v("s"))
D("encK", ["s","k","i","z"], v("k"))
D("encI", ["s","k","i","z"], v("i"))
D("encA", ["t","u","s","k","i","z"], A(v("z"), v("t"), v("u")))
mkApp = a("encA")
def enc(t):
    L = {"S":"encS","K":"encK","I":"encI"}
    out, work = [], [(t, False)]
    while work:
        x, done = work.pop()
        if isinstance(x, Atom): out.append(a(L[x.name])); continue
        if not done:
            work.append((x, True)); work.append((x.arg, False)); work.append((x.fn, False))
        else:
            r = out.pop(); l = out.pop(); out.append(A(mkApp, l, r))
    return out.pop()
for nm, e in [("<S>", enc(S)), ("<K>", enc(K)), ("<I>", enc(I))]:
    print(f"  {nm} c1 c2 c3 c4  ->", pretty(run(A(e, v("c1"),v("c2"),v("c3"),v("c4")), whnf=False).term))
print("  <t u> c1 c2 c3 c4 ->", pretty(run(A(enc(App(S,K)), v("c1"),v("c2"),v("c3"),v("c4")), whnf=False).term))

  <S> c1 c2 c3 c4  -> c1
  <K> c1 c2 c3 c4  -> c2
  <I> c1 c2 c3 c4  -> c3
  <t u> c1 c2 c3 c4 -> c4 encS encK


## 4. What the encoding costs

Encoding is not free, and it is not a function *in* SKI — quoting is
necessarily a meta-level operation, for exactly the reason above. That is why
`enc` below is written in Python: it is the one step that must come from
outside.

In [4]:
print(f"  S K K      {size(skk):>3d} atoms")
print(f"  <S K K>    {size(expand(enc(skk), env)):>3d} atoms")
print(f"  I          {size(I):>3d} atoms")
print(f"  <I>        {size(expand(enc(I), env)):>3d} atoms")

  S K K        3 atoms
  <S K K>     81 atoms
  I            1 atoms
  <I>          3 atoms


## 5. Scott data, and the unquoter

`UQ` interprets each constructor as *itself* and application as application,
so it releases the encoded term back into the host: `UQ <t> →* t`. It is the
left inverse of encoding — a **self-recogniser**, in Jay & Palsberg's
terminology — and it will double as our decoder. 43 atoms.

In [5]:
D("nil",["n","c"], v("n")); D("cons",["h","t","n","c"], A(v("c"),v("h"),v("t")))
D("nothing",["n","j"], v("n")); D("just",["x","n","j"], A(v("j"),v("x")))
D("zero",["z","sc"], v("z")); D("suc",["n","z","sc"], A(v("sc"),v("n")))
def nat(k):
    t = a("zero")
    for _ in range(k): t = A(a("suc"), t)
    return t
D("UQA",["f","t","u"], A(A(v("f"),v("t")), A(v("f"),v("u"))))
D("UQGen",["f","m"], A(v("m"), S, K, I, A(a("UQA"), v("f"))))
AL("UQ", A(Y, a("UQGen")))
for t in [S, App(K,S), A(S,K,K), A(S,I,I,K)]:
    got = run(A(a("UQ"), enc(t)), whnf=False).term
    ref = run(t, whnf=False).term
    print(f"  UQ <{pretty(t):8s}> -> {pretty(got):10s} | direct {pretty(ref):10s} {'OK' if pretty(got)==pretty(ref) else 'FAIL'}")

  UQ <S       > -> S          | direct S          OK
  UQ <K S     > -> K S        | direct K S        OK
  UQ <S K K   > -> S K K      | direct S K K      OK
  UQ <S I I K > -> K K        | direct K K        OK


## 6. Taking a term apart

`sp` unwinds the spine, collecting arguments into a list; `rb` rebuilds an
application from a head and a list. `step` performs one normal-order
contraction, returning `just <t'>` if it fired and `nothing` if the head is
under-applied. This is where the encoding earns its keep: the four `step`
arms are ordinary case analysis on a data structure.

In [6]:
D("resS",["acc","s","k","i"], A(v("s"), v("acc")))
D("resK",["acc","s","k","i"], A(v("k"), v("acc")))
D("resI",["acc","s","k","i"], A(v("i"), v("acc")))
D("spApp",["f","acc","t","u"], A(v("f"), v("t"), A(a("cons"), v("u"), v("acc"))))
D("spGen",["f","m","acc"], A(v("m"), A(a("resS"),v("acc")), A(a("resK"),v("acc")),
                                     A(a("resI"),v("acc")), A(a("spApp"),v("f"),v("acc"))))
AL("sp", A(Y, a("spGen")))
D("rb1",["f","h","x","xs"], A(v("f"), A(mkApp, v("h"), v("x")), v("xs")))
D("rbGen",["f","h","args"], A(v("args"), v("h"), A(a("rb1"), v("f"), v("h"))))
AL("rb", A(Y, a("rbGen")))
D("stepI1",["x","rest"], A(a("just"), A(a("rb"), v("x"), v("rest"))))
D("stepI",["args"], A(v("args"), a("nothing"), a("stepI1")))
D("stepK2",["x","y","rest"], A(a("just"), A(a("rb"), v("x"), v("rest"))))
D("stepK1",["x","r"], A(v("r"), a("nothing"), A(a("stepK2"), v("x"))))
D("stepK",["args"], A(v("args"), a("nothing"), a("stepK1")))
D("stepS3",["x","y","z","rest"], A(a("just"), A(a("rb"),
    A(mkApp, A(mkApp,v("x"),v("z")), A(mkApp,v("y"),v("z"))), v("rest"))))
D("stepS2",["x","y","r2"], A(v("r2"), a("nothing"), A(a("stepS3"), v("x"), v("y"))))
D("stepS1",["x","r"], A(v("r"), a("nothing"), A(a("stepS2"), v("x"))))
D("stepS",["args"], A(v("args"), a("nothing"), a("stepS1")))
D("step",["m"], A(a("sp"), v("m"), a("nil"), a("stepS"), a("stepK"), a("stepI")))
for t in [App(I,K), A(K,S,I), A(S,K,K,I)]:
    r = run(A(a("step"), enc(t)))
    print(f"  step <{pretty(t):8s}> -> {pretty(run(A(r.term, v('NO'), v('JU')), whnf=False).term)}")

  step <I K     > -> JU encK
  step <K S I   > -> JU encS
  step <S K K I > -> JU (encA (encA encK encI) (encA encK encI))


## 7. The fueled loop

`whnfF n <t>` performs up to `n` object-level steps, returning `just <v>` on
reaching weak head normal form and `nothing` on exhaustion. Fuel is what makes
the interpreter total, and it introduces a third outcome — the object program
can now *time out* as well as return.

To read a result we apply it to two continuations: `TIMEOUT` for `nothing`,
and `UQ` for `just`. Dispatch and decode in one application.

In [7]:
D("wf1",["w","m","n2"], A(a("step"), v("m"), A(a("just"), v("m")), A(v("w"), v("n2"))))
D("wfGen",["w","n","m"], A(v("n"), a("nothing"), A(a("wf1"), v("w"), v("m"))))
AL("whnfF", A(Y, a("wfGen")))
def decode(res, fuel=10**7):
    return run(A(res, v("TIMEOUT"), a("UQ")), whnf=False, fuel=fuel)
for t, k in [(A(S,K,K,I), 20), (A(S,I,I,K), 20), (A(K,I,App(S,I)), 20)]:
    r = run(A(a("whnfF"), nat(k), enc(t)), fuel=10**7)
    d = decode(r.term)
    ref = run(t, whnf=False)
    print(f"  whnfF {k} <{pretty(t):10s}> -> {pretty(d.term):8s} | direct {pretty(ref.term):8s} "
          f"{'OK' if pretty(d.term)==pretty(ref.term) else 'FAIL'}  ({r.steps} contractions)")

  whnfF 20 <S K K I   > -> I        | direct I        OK  (76 contractions)
  whnfF 20 <S I I K   > -> K K      | direct K K      OK  (84 contractions)
  whnfF 20 <K I (S I) > -> I        | direct I        OK  (42 contractions)


## 8. In pure {S, K, I}

Everything above was written with named definitions for readability. Expanding
them gives closed combinator terms over the basis alone.

In [8]:
WP = expand(a("whnfF"), env); UP = expand(a("UQ"), env)
encP = lambda t: expand(enc(t), env)
natP = lambda k: expand(nat(k), env)
def atoms(t):
    st, out = [t], set()
    while st:
        x = st.pop()
        if isinstance(x, Atom): out.add(x.name)
        else: st.append(x.fn); st.append(x.arg)
    return out
print(f"  whnfF  {size(WP):>4d} atoms   basis {sorted(atoms(WP))}")
print(f"  UQ     {size(UP):>4d} atoms   basis {sorted(atoms(UP))}")
t = App(I,K)
r = run(A(WP, natP(3), encP(t)), fuel=10**7)
print(f"  pure whnfF 3 <I K>: {r.steps} contractions -> "
      f"{pretty(run(A(r.term, v('TIMEOUT'), UP), whnf=False).term)}")

  whnfF   618 atoms   basis ['I', 'K', 'S']
  UQ       43 atoms   basis ['I', 'K', 'S']
  pure whnfF 3 <I K>: 340 contractions -> K


## 9. Divergence, and what fuel can and cannot tell you

`Ω = (S I I)(S I I)` has no normal form. `whnfF k <Ω>` reports `TIMEOUT` at
every `k` — but so does `whnfF k <I³¹ K>`, which converges at 32.

**A finite run cannot establish divergence, only refute it.** This is why the
outcome set is `{value, timeout}` and not `{value, divergence}`: timeout is an
observation, divergence is a claim about all observations.

In [9]:
OM = A(A(S,I,I), A(S,I,I))
def nestI(n):
    t = K
    for _ in range(n): t = App(I, t)
    return t
print("  k    whnfF k <Omega>   whnfF k <I^31 K>")
for k in [1, 10, 31, 32]:
    g1 = run(A(a("whnfF"), nat(k), enc(OM)), fuel=10**7)
    g2 = run(A(a("whnfF"), nat(k), enc(nestI(31))), fuel=10**7)
    d1 = pretty(decode(g1.term).term); d2 = pretty(decode(g2.term).term)
    print(f"  {k:<4d} {d1:<16s} {d2}")

  k    whnfF k <Omega>   whnfF k <I^31 K>
  1    TIMEOUT          TIMEOUT
  10   TIMEOUT          TIMEOUT
  31   TIMEOUT          TIMEOUT
  32   TIMEOUT          K


## 10. Object-level laziness survives

`K I Ω` discards `Ω` without evaluating it. The emulator inherits this: it
never touches the argument, because the object-level `K` rule discards it
before the interpreter recurses.

In [10]:
g = run(A(a("whnfF"), nat(30), enc(A(K, I, OM))), fuel=10**7)
print("  whnfF 30 <K I Omega> ->", pretty(decode(g.term).term))

  whnfF 30 <K I Omega> -> I


## 11. The tower

`whnfF` is a program. Encode it and feed it to itself.

Note what comes back. The outer interpreter returns the encoded *result term*
of the inner run — `just <K>` — not the value `K`. A self-interpreter that
maps a representation to a value is a **recogniser**; one that maps a
representation to the representation of a value is an **evaluator**. `UQ` is
the first, `whnfF` the second, and stacking them makes the difference visible.

(~2s.)

In [11]:
inner = A(WP, natP(3), encP(App(I,K)))
print(f"  inner   whnfF 3 <I K>          {size(inner):>7,} atoms")
lvl2 = A(WP, natP(360), encP(inner))
print(f"  outer   whnfF 360 <inner>      {size(lvl2):>7,} atoms")
t0 = time.time(); r2 = run(lvl2, fuel=10**8); el = time.time()-t0
print(f"  reduced: {r2.steps:,} contractions in {el:.0f}s")
d2 = run(A(r2.term, v("TIMEOUT"), UP), whnf=False, fuel=10**8)
ref = run(inner, whnf=False, fuel=10**7)          # level 1 run directly
print(f"  decoded  : {pretty(d2.term)[:44]}")
print(f"  level 1  : {pretty(ref.term)[:44]}")
print(f"  agree    : {pretty(d2.term)==pretty(ref.term)}")
print("  note: the tower returns the encoded level-1 RESULT TERM (`just <K>`),")
print("  not the final value K -- an evaluator, not a recogniser.")

  inner   whnfF 3 <I K>              683 atoms
  outer   whnfF 360 <inner>       29,244 atoms


  reduced: 504,930 contractions in 1s
  decoded  : K (S I (K (K (S (K K) K))))
  level 1  : K (S I (K (K (S (K K) K))))
  agree    : True
  note: the tower returns the encoded level-1 RESULT TERM (`just <K>`),
  not the final value K -- an evaluator, not a recogniser.


## 12. Adding an operator, and who decides what it means

Extend the object language with a fifth constructor `Err` — no reduction
rules, like a constructor in any algebraic data type. Its encoding is

    <Err> = K (K (K (K I)))

five atoms of pure `{S, K, I}`. Apply it to five continuations and it selects
the fifth. **It carries no semantics whatsoever.** It cannot crash, diverge,
or absorb; it is inert.

So what does `Err` *mean*? Whatever the interpreter's arm says it means. Here
are two interpreters, identical except for that one arm:

    errArmAbs = K (K (K I))          4 atoms   -- expose: crash becomes a value
    errArmOmg = K (S I I (S I I))    7 atoms   -- hide: crash becomes divergence

Three atoms apart, in interpreters of 768 and 771 atoms. Read the table below:
the two agree everywhere except where `Err` is actually *reached*. `K I Err`
discards it, so laziness hides the choice; `Ω` diverges for reasons of its own.

The encoding of a failure carries no failure. The semantics is authored, at a
named site, and it has a measurable price.

In [12]:
D("encS5",["s","k","i","z","e"], v("s")); D("encK5",["s","k","i","z","e"], v("k"))
D("encI5",["s","k","i","z","e"], v("i")); D("encE5",["s","k","i","z","e"], v("e"))
D("encA5",["t","u","s","k","i","z","e"], A(v("z"), v("t"), v("u")))
mk5 = a("encA5")
def enc5(t):
    L = {"S":"encS5","K":"encK5","I":"encI5","Err":"encE5"}
    out, work = [], [(t, False)]
    while work:
        x, dn = work.pop()
        if isinstance(x, Atom): out.append(a(L[x.name])); continue
        if not dn:
            work.append((x, True)); work.append((x.arg, False)); work.append((x.fn, False))
        else:
            r = out.pop(); l = out.pop(); out.append(A(mk5, l, r))
    return out.pop()
print("  <Err> =", pretty(expand(a("encE5"), env)), f"  ({size(expand(a('encE5'), env))} atoms)")
print("  <Err> c1 c2 c3 c4 c5 ->",
      pretty(run(A(expand(a("encE5"),env), v("c1"),v("c2"),v("c3"),v("c4"),v("c5")), whnf=False).term))

D("r5S",["acc","s","k","i","e"], A(v("s"),v("acc")))
D("r5K",["acc","s","k","i","e"], A(v("k"),v("acc")))
D("r5I",["acc","s","k","i","e"], A(v("i"),v("acc")))
D("r5E",["acc","s","k","i","e"], A(v("e"),v("acc")))
D("sp5App",["f","acc","t","u"], A(v("f"),v("t"),A(a("cons"),v("u"),v("acc"))))
D("sp5Gen",["f","m","acc"], A(v("m"), A(a("r5S"),v("acc")),A(a("r5K"),v("acc")),
                              A(a("r5I"),v("acc")),A(a("sp5App"),v("f"),v("acc")),A(a("r5E"),v("acc"))))
AL("sp5", A(Y, a("sp5Gen")))
D("rb51",["f","h","x","xs"], A(v("f"), A(mk5,v("h"),v("x")), v("xs")))
D("rb5Gen",["f","h","args"], A(v("args"), v("h"), A(a("rb51"),v("f"),v("h"))))
AL("rb5", A(Y, a("rb5Gen")))
D("stepped",["t","c1","c2","c3"], A(v("c1"),v("t")))
D("done",["c1","c2","c3"], v("c2")); D("errd",["c1","c2","c3"], v("c3"))
D("rVal",["x","c1","c2","c3"], A(v("c1"),v("x")))
D("rErr",["c1","c2","c3"], v("c2")); D("rTime",["c1","c2","c3"], v("c3"))
D("q5I1",["x","rest"], A(a("stepped"), A(a("rb5"),v("x"),v("rest"))))
D("q5I",["args"], A(v("args"), a("done"), a("q5I1")))
D("q5K2",["x","y","rest"], A(a("stepped"), A(a("rb5"),v("x"),v("rest"))))
D("q5K1",["x","r"], A(v("r"), a("done"), A(a("q5K2"),v("x"))))
D("q5K",["args"], A(v("args"), a("done"), a("q5K1")))
D("q5S3",["x","y","z","rest"], A(a("stepped"), A(a("rb5"),
    A(mk5, A(mk5,v("x"),v("z")), A(mk5,v("y"),v("z"))), v("rest"))))
D("q5S2",["x","y","r2"], A(v("r2"), a("done"), A(a("q5S3"),v("x"),v("y"))))
D("q5S1",["x","r"], A(v("r"), a("done"), A(a("q5S2"),v("x"))))
D("q5S",["args"], A(v("args"), a("done"), a("q5S1")))
OMEGA = A(A(A(S,I),I), A(A(S,I),I))
D("errArmAbs",["acc"], a("errd"))          # expose: crash becomes a value
D("errArmOmg",["acc"], OMEGA)              # hide: crash becomes divergence
for tag in ("Abs","Omg"):
    D(f"st5{tag}",["m"], A(a("sp5"),v("m"),a("nil"),a("q5S"),a("q5K"),a("q5I"),a(f"errArm{tag}")))
    D(f"wf5{tag}1",["w","m","n2"], A(a(f"st5{tag}"),v("m"), A(v("w"),v("n2")),
                                      A(a("rVal"),v("m")), a("rErr")))
    D(f"wf5{tag}Gen",["w","n","m"], A(v("n"), a("rTime"), A(a(f"wf5{tag}1"),v("w"),v("m"))))
    AL(f"wf5{tag}", A(Y, a(f"wf5{tag}Gen")))
D("decodeTag",["r"], A(v("r"), A(K, v("VAL")), v("ERR"), v("TIME")))
WA = expand(a("wf5Abs"), env); WO = expand(a("wf5Omg"), env); DT = expand(a("decodeTag"), env)
print(f"\n  errArmAbs = {pretty(expand(a('errArmAbs'),env)):<22s} {size(expand(a('errArmAbs'),env))} atoms")
print(f"  errArmOmg = {pretty(expand(a('errArmOmg'),env)):<22s} {size(expand(a('errArmOmg'),env))} atoms")
print(f"  interpreters: {size(WA)} vs {size(WO)} atoms\n")
Err = Atom("Err")
print(f"  {'object term':<16s}{'expose':>10s}{'hide':>10s}")
for lbl, ot in [("Err", Err), ("Err K", App(Err,K)), ("K I Err", A(K,I,Err)),
                ("I K", App(I,K)), ("Omega", OMEGA)]:
    cells = []
    for Wx in (WA, WO):
        rr = run(A(DT, A(Wx, natP(20), expand(enc5(ot), env))), whnf=False, fuel=400_000)
        cells.append(pretty(rr.term) if rr.status in (Status.NORMAL, Status.WHNF) else "DIVERGES")
    print(f"  {lbl:<16s}{cells[0]:>10s}{cells[1]:>10s}")

  <Err> = K (K (K (K I)))   (5 atoms)
  <Err> c1 c2 c3 c4 c5 -> c5

  errArmAbs = K (K (K I))            4 atoms
  errArmOmg = K (S I I (S I I))      7 atoms
  interpreters: 768 vs 771 atoms

  object term         expose      hide


  Err                    ERR  DIVERGES


  Err K                  ERR  DIVERGES
  K I Err                VAL       VAL
  I K                    VAL       VAL
  Omega                 TIME      TIME


## Where this sits

The artifact is 618 atoms of `{S, K, I}`, and it emulates itself in 504,930
contractions. What it demonstrates is not that self-interpretation is possible
— Kleene had that in 1936, Mogensen made it efficient in 1992, and Jay &
Palsberg did it for a combinator calculus with types in 2011 — but what it
*costs* when you refuse to extend the calculus.

Three things are worth carrying away.

**Encoding is external; decoding is internal.** Quoting must separate terms
that are extensionally equal, which SKI cannot do. Unquoting is 43 atoms of
ordinary SKI. The asymmetry is forced.

**Fuel buys a third outcome.** An unfueled interpreter has `{value,
divergence}`, and divergence is not observable. Fuel converts it to `{value,
timeout}`, both of which are values in the host — and that is what makes the
next point expressible at all.

**Adding a constructor does not add a meaning.** `<Err>` is a selector, and a
selector says nothing. Whoever writes the interpreter decides whether that
outcome becomes visible or vanishes into divergence, and in this substrate the
entire decision is four atoms against seven.

### References

- C. B. Jay and T. Given-Wilson, *A combinatory account of internal
  structure*, J. Symbolic Logic 76(3):807–826, 2011.
- C. B. Jay and J. Palsberg, *Typed self-interpretation by pattern matching*,
  ICFP 2011, 247–258.
- T. Æ. Mogensen, *Efficient self-interpretations in lambda calculus*,
  JFP 2(3):345–363, 1992.
- M. Wand, *The theory of fexprs is trivial*, LISP and Symbolic Computation
  10:189–199, 1998.